# CS781 Phase 4 (CJPE) — v2: fine-tune ALL Phase 1 models, ensemble them, and predict (built from the weaknesses seen in the first GPT-2 run)

**First run (for reference):** GPT-2, last 512 tokens, 1.3 epochs: dev macro-F1 0.671, leaderboard 0.6613 (the dev set is a trustworthy proxy: gap 0.010). Its weaknesses, and what this notebook does about each:

| weakness seen in the first run | change here |
|---|---|
| under-trained: 1.3 epochs, dev score still rising at the end | **3 epochs**, weight averaging (**EMA**) and the final EMA weights are used (no noisy "best snapshot" choice) |
| read only the last 512 tokens of cases whose median is about 4,000 tokens | **multi-window**: training samples a random window from the last `W` blocks of the case (half the time the very end), inference scores all `W` windows and averages them with a recency decay chosen on dev |
| the hierarchical model was starved (0.56 epochs) and could not be judged | no end-to-end hierarchy: windows cost the same per step as the plain model. **X1 vs X2 is a controlled A/B** (random windows vs last window only), so the idea is measured, not assumed |
| one backbone only | the same recipe for the GPT-2, the encoder, the T5 encoder, the BiLSTM and the old GPT-2 (found automatically); `lr_pilot()` picks each new backbone's learning rate on dev AUC |
| the dev-tuned threshold gave too few positives on the test (about 42% predicted against about 50%) | the submission uses **rate matching**: the top share of cases (the dev positive rate) is flagged 1. It needs scores only, never labels of the set being predicted. The dev-tuned alternative is also reported |
| 2-model ensemble of near-identical runs | ensemble of standardised margins across different architectures; the number of models is chosen on **dev AUC**, preferring more models when within 0.002 |
| test.json contains a `label` field | it is **ignored and blanked** in cell 3 (the task says test labels are private) |

## Guardrails against overfitting the leaderboard
Choose everything on **dev** (3,505 cases, balanced, similar to the test set). Look at the leaderboard once per final candidate. One standard error on 1,517 test cases is about 0.012, so differences below about 0.02 are noise. The ensemble is only used when it beats the best single run on dev AUC by at least 0.002.

## Run order and time (estimates from the first run's speed: 0.6 s per 16 cases for GPT-2 at 512 tokens)
1 setup -> 2 Hugging Face access check (skip if the CJPE data is attached; in a committed run use a Kaggle secret `HF_TOKEN`) -> 3 data (first run tokenises 850 MB on the CPU, about 10 min) -> 4 models -> 5 trainer -> **6 smoke test** -> then the experiments in value order:
**X1** (GPT-2, windows, about 80 min) -> **X2** (control, about 80 min) -> encoder (pilot 30 min + about 70 min) -> T5 encoder (same) -> BiLSTM (same) -> X6 old GPT-2 (about 60 min) -> final ensemble (cell 8) -> diagnostics (cell 9). Optional: X1b (mean+last pooling, about 80 min).
Each experiment adds a row to the ledger; the final cell uses whatever is there (it needs six runs).

## What was tested (CPU, tiny models, synthetic data in the real file layout) and what was not
AUC matches the pair-counting definition; rate matching flags exactly the top share; window averaging with missing windows; windows tile the end of each case; EMA apply/restore; backbone dropout; test labels blanked; all four architectures and the old-format checkpoint train end to end; `lr_pilot`; the six-file packaging, the ensemble choice and the diagnostics. **A planted signal 80-110 tokens before the end was found by the multi-window model (AUC 0.998) and missed by the tail-only control (0.518).**
**Not tested:** the real CJPE files, GPU speed, and whether multi-window training helps on real data (that is exactly what X1 vs X2 will show). Expect gains of a few points, not a jump to the top of the leaderboard: the first-place score (0.8424) is above the published best for this corpus (about 78% macro-F1).


In [ ]:
# ================= CELL 1 — setup =================
import os, sys, re, json, math, time, random, glob, gc, copy, hashlib, pickle, getpass, zipfile
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda"
WORK = Path("/kaggle/working/cjpe"); (WORK / "runs").mkdir(parents=True, exist_ok=True); (WORK / "cache").mkdir(exist_ok=True); SUBDIR = WORK / "submissions"; SUBDIR.mkdir(exist_ok=True)
LEDGER = WORK / "ledger.csv"
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu (training cells need a GPU)", "| work dir:", WORK)


In [ ]:
# ===== HUGGING FACE ACCESS CHECK — run this BEFORE cell 2 (CPU session is enough). It tells you exactly why the download is refused, and on success hands the verified token to cell 2. =====
import os, getpass, shutil
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
REPO, TEST_FILE = "Exploration-Lab/CS-781-Capstone", "Phase_4_CJPE/Test/test.json"; LOCAL = "/kaggle/working/cjpe/hf"
for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
    if os.environ.get(k): print(f"note: environment variable {k} is set ({len(os.environ[k])} characters) and would override a Kaggle secret; this cell replaces it")
token = getpass.getpass("Hugging Face token (hidden as you type; paste it exactly, no quotes): ").strip().strip("'\"")
print(f"token length {len(token)} | starts with 'hf_': {token.startswith('hf_')}")
api = HfApi(); ok = False
try:
    me = api.whoami(token=token); tk = (me.get("auth") or {}).get("accessToken") or {}
    print(f"1) token ACCEPTED by Hugging Face: user '{me.get('name')}' | token type: {tk.get('role', 'unknown')}" + (f" | fine-grained permissions: {tk.get('fineGrained')}" if tk.get("role") == "fineGrained" else ""))
    if tk.get("role") == "fineGrained": print("   NOTE: a fine-grained token must include 'Read access to contents of all public gated repos you can access'; the simplest fix is a classic 'Read' token (huggingface.co/settings/tokens)")
    ok = True
except Exception as e:
    code = getattr(getattr(e, "response", None), "status_code", None)
    print(f"1) token REJECTED (HTTP {code}). Hugging Face does not recognise this token: a typo or extra characters, a revoked or expired token, or the wrong value saved as the secret. Create a NEW classic 'Read' token at huggingface.co/settings/tokens and paste it again.")
if ok:
    try:
        api.auth_check(REPO, repo_type="dataset", token=token); print(f"2) access to {REPO}: OK")
    except Exception as e:
        code = getattr(getattr(e, "response", None), "status_code", None); ok = False
        print(f"2) access to {REPO}: REFUSED (HTTP {code}, {type(e).__name__}). The token is valid but THIS account has not been approved for the gated dataset.")
        print(f"   Log in to huggingface.co as '{me.get('name')}', open https://huggingface.co/datasets/{REPO}, accept the conditions / request access, and wait until the page shows you have access. (The data you saw in the web viewer may have been under a different account.)")
if ok:
    try:
        p = hf_hub_download(REPO, TEST_FILE, repo_type="dataset", token=token, local_dir=LOCAL); print(f"3) real download works: {p} ({os.path.getsize(p)/2**20:.0f} MiB)")
        os.environ["HF_TOKEN"] = token; print("\nALL GOOD: the token is stored for this session only; now run cell 2 (it will use it and download the other two files).")
    except Exception as e: ok = False; print(f"3) download FAILED: {type(e).__name__}: {str(e)[:300]}")
if not ok: print("\nIf you cannot get access in time: download train.json, dev.json and test.json in the browser from the dataset page (while logged in with access), upload them as a Kaggle dataset whose name contains 'cjpe' (for example cs781-phase4-cjpe-data), attach it, and run cell 2: it finds the files by name.")
del token


In [ ]:
# ================= CELL 2 — CJPE data: locate (or download), detect the schema, tokenise ONLY what is needed (the end of each case), cache =================
HF_REPO = "Exploration-Lab/CS-781-Capstone"          # gated: Phase_4_CJPE/Train/train.json (737 MB), Train/dev.json (80 MB), Test/test.json (35 MB)
TAIL_MAX = 4096                                      # tokens kept from the END of every case (the end of a judgment is the most informative part)
TEXT_KEYS = ("text", "document", "case_text", "content", "judgment", "doc", "full_text", "case"); LABEL_KEYS = ("label", "labels", "decision", "outcome", "y", "gold", "judgement", "verdict")
ID_KEYS = ("id", "case_id", "doc_id", "document_id", "name", "file", "filename", "index")

def locate():
    got = {}
    for sp in ("train", "dev", "test"):
        c = [p for root in SEARCH_ROOTS for p in glob.glob(f"{root}/**/{sp}.json", recursive=True) if re.search(r"cjpe|phase[_ -]?4", p, re.I)]
        if c: got[sp] = max(c, key=os.path.getsize)
    return got
paths = locate()
if len(paths) < 3:
    print("CJPE files not found locally -> downloading from Hugging Face (accept the dataset conditions on its page first)")
    from huggingface_hub import snapshot_download
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception: token = getpass.getpass("Hugging Face token (hidden; not stored): ").strip()
    local = snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["Phase_4_CJPE/*"], token=token, local_dir=str(WORK / "hf")); SEARCH_ROOTS.append(local); paths = locate()
assert len(paths) == 3, f"need train/dev/test json, found {paths}"
print({k: f"{os.path.getsize(v)/2**20:.0f} MiB  {v}" for k, v in paths.items()})

def normalise(obj):
    if isinstance(obj, dict):
        for wk in ("data", "documents", "examples", "cases", "items", "train", "dev", "test", "validation"):
            if wk in obj and isinstance(obj[wk], (list, dict)) and len(obj) <= 4: return normalise(obj[wk])
        out = []
        for k, v in obj.items():
            if isinstance(v, dict): r = dict(v); r.setdefault("id", k)
            elif isinstance(v, str): r = {"id": k, "text": v}
            else: continue
            out.append(r)
        return out
    return [r for r in obj if isinstance(r, dict)]
def detect(recs):
    keys = list(recs[0].keys()); strlen = {k: np.mean([len(str(r.get(k, ""))) for r in recs[:200]]) for k in keys if isinstance(recs[0].get(k), str)}
    tk = next((k for k in TEXT_KEYS if k in keys), None) or (max(strlen, key=strlen.get) if strlen else None); lk = next((k for k in LABEL_KEYS if k in keys), None)
    if lk is None:
        for k in keys:
            if k == tk: continue
            vals = {str(r.get(k)) for r in recs[:2000] if r.get(k) is not None}
            if 1 < len(vals) <= 5 and k not in ID_KEYS: lk = k; break
    return tk, lk, next((k for k in ID_KEYS if k in keys), None)
def binary_map(vals):
    s = {str(v).strip().lower() for v in vals if v is not None}
    if s <= {"0", "1", "0.0", "1.0", "true", "false"}: return lambda v: int(str(v).strip().lower() in ("1", "1.0", "true"))
    raise ValueError(f"label values {sorted(s)[:10]} are not 0/1: tell me what they mean (for example accepted/rejected) before training")

from tokenizers import Tokenizer
tpaths = sorted(glob.glob("/kaggle/input/**/legal_bpe_32k_mask.json", recursive=True)) or sorted(glob.glob("/kaggle/input/**/legal_bpe_32k*.json", recursive=True)) or sorted(glob.glob("/kaggle/working/**/legal_bpe_32k*.json", recursive=True))
assert tpaths, "attach the tokenizer file legal_bpe_32k_mask.json (it is inside your Phase 1 dataset)"
tok = Tokenizer.from_file(tpaths[0]); EOS, BOS, PAD = tok.token_to_id("[EOS]"), tok.token_to_id("[BOS]"), tok.token_to_id("[PAD]"); print("tokenizer:", tpaths[0], "| vocabulary", tok.get_vocab_size())

def prepare(split, path, bs=256):
    cache = WORK / "cache" / f"{split}.pkl"
    if cache.exists():
        d = pickle.load(open(cache, "rb")); print(f"[{split}] loaded cache: {len(d['tails'])} cases"); return d
    print(f"[{split}] reading {path} ..."); obj = json.load(open(path, encoding="utf-8")); recs = normalise(obj); del obj; gc.collect()
    tk, lk, ik = detect(recs); lmap = binary_map([r.get(lk) for r in recs[:5000]]) if lk else None
    print(f"[{split}] {len(recs)} cases | text field '{tk}' | label field '{lk}' | id field '{ik}'")
    tails, lens, labels, rids = [], [], [], []
    for i in range(0, len(recs), bs):
        chunk = recs[i:i + bs]
        for e in tok.encode_batch([str(r.get(tk, "")) for r in chunk], add_special_tokens=False):
            lens.append(len(e.ids)); tails.append(np.asarray((e.ids + [EOS])[-TAIL_MAX:], dtype=np.int32))
        labels += [lmap(r.get(lk)) if lmap else -1 for r in chunk]; rids += [r.get(ik, i + j) if ik else i + j for j, r in enumerate(chunk)]
        for r in chunk: r.clear()
    if split == "test":
        if lk and any(l >= 0 for l in labels): print("[test] NOTE: test.json contains a label field. The task says test labels are private, so they are IGNORED here and never used for anything.")
        labels = [-1] * len(labels)
    d = dict(tails=tails, lens=np.asarray(lens), labels=np.asarray(labels), rids=rids, keys=dict(text=tk, label=lk, id=ik)); pickle.dump(d, open(cache, "wb"), protocol=4); del recs; gc.collect(); return d
DATA = {sp: prepare(sp, p) for sp, p in paths.items()}
DATA["test"]["labels"] = np.full(len(DATA["test"]["tails"]), -1)       # test labels are private: blank them even if a cache from an earlier run held them
for sp, d in DATA.items():
    y = d["labels"]; print(f"[{sp}] cases {len(d['tails']):6d} | tokens p50/p90/p99: {int(np.percentile(d['lens'],50))}/{int(np.percentile(d['lens'],90))}/{int(np.percentile(d['lens'],99))} | label counts: {dict(Counter(y.tolist())) if (y>=0).any() else 'none (test)'}")
assert len(set(map(str, DATA["test"]["rids"]))) == len(DATA["test"]["rids"]), "duplicate test ids"
ytr = DATA["train"]["labels"]; print(f"majority-class baseline macro-F1 on dev would be about {0.5 * (2*max(np.mean(DATA['dev']['labels']==0), np.mean(DATA['dev']['labels']==1)) / (1+max(np.mean(DATA['dev']['labels']==0), np.mean(DATA['dev']['labels']==1)))):.3f}")


In [ ]:
# ================= CELL 3 — load Phase 1 models (all four architectures + the old GPT-2), document classifier, chunked batching =================
al_paths = sorted(glob.glob("/kaggle/input/**/arch_loaders.py", recursive=True)) or sorted(glob.glob("/kaggle/working/**/arch_loaders.py", recursive=True))
assert al_paths, "attach the dataset/notebook output that contains arch_loaders.py (written by cell 8 of the pre-training notebooks)"
sys.path.insert(0, str(Path(al_paths[0]).parent)); import arch_loaders as AL

MODEL_PATHS = {}                                    # filled automatically below; you can also set entries by hand, e.g. MODEL_PATHS["bilstm"] = "/kaggle/input/.../final_model.pt"
def discover_models():
    found = {}
    for p in sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)):
        if os.path.getsize(p) < 50e6: continue
        base = os.path.basename(p)
        if base != "final_model.pt" and not (base == "checkpoint_latest.pt" and "phase-1" in p): continue
        try: ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
        except Exception: continue
        if not isinstance(ck, dict) or "model_config" not in ck or "model_state_dict" not in ck: continue
        mc = ck["model_config"]; fam = mc.get("family", "gpt2"); old = (base == "checkpoint_latest.pt")
        key = "gpt2_old" if old else ("gpt2_new" if fam == "gpt2" else fam)
        if key not in found: found[key] = p
    return found
MODEL_PATHS.update({k: v for k, v in discover_models().items() if k not in MODEL_PATHS})
print("Phase 1 models found:", {k: v for k, v in MODEL_PATHS.items()}, "\n  missing (pre-train them first):", [k for k in ("gpt2_new", "modernbert", "t5", "bilstm", "gpt2_old") if k not in MODEL_PATHS])

def load_backbone_any(path):
    """like AL.load_backbone, but also accepts the OLD Phase 1 checkpoint (its model_config has no 'family' entry)"""
    ck = torch.load(path, map_location="cpu", weights_only=False); mc = dict(ck["model_config"]); mc.setdefault("family", "gpt2")
    full = AL.build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True); f = mc["family"]
    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte
    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings
    elif f == "t5": inner, embed = full.encoder, full.shared
    elif f == "bilstm": inner, embed = full, full.emb
    else: raise ValueError(f)
    del ck; return AL.Backbone(f, inner, embed), mc["hidden_size"], AL.CAUSAL[f], mc["context_length"], f

class DocClassifier(nn.Module):
    """Encodes K chunks of a case (the last K*L tokens, right-aligned) with a Phase 1 backbone, pools each chunk, combines the chunks with attention pooling
       + the final chunk, and classifies. K=1 is the plain 'last L tokens' model. Chunks that are empty (short cases) are skipped, never fed to the backbone."""
    def __init__(self, backbone, hidden, causal, K, dropout=0.1, pool="last"):
        super().__init__(); self.backbone, self.hidden, self.causal, self.K, self.pool = backbone, hidden, causal, K, pool
        self.chunk_pos = nn.Parameter(torch.zeros(K, hidden)); self.proj = nn.Linear(hidden, hidden); self.att = nn.Linear(hidden, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 2))
    def encode_chunks(self, ids, mask):
        B, K, L = ids.shape; valid = mask.sum(-1) > 0; fi, fm = ids[valid], mask[valid]; T = int(fm.sum(1).max()); fi, fm = fi[:, :T], fm[:, :T]
        h = self.backbone(input_ids=fi, attention_mask=fm).last_hidden_state
        mm = fm.unsqueeze(-1).to(h.dtype); mean = (h * mm).sum(1) / mm.sum(1).clamp(min=1)
        if self.causal:
            last = h[torch.arange(h.shape[0], device=h.device), fm.sum(1) - 1]; pooled = last if self.pool == "last" else 0.5 * (last + mean)      # pool='meanlast': average of the last-token and mean-pooled vectors
        else: pooled = mean
        out = pooled.new_zeros(B, K, self.hidden); out[valid] = pooled; return out, valid
    def forward(self, ids, mask):
        x, valid = self.encode_chunks(ids, mask); x = x + self.chunk_pos.to(x.dtype)
        s = self.att(torch.tanh(self.proj(x))).squeeze(-1).float().masked_fill(~valid, -1e4); w = torch.softmax(s, -1).to(x.dtype)
        return self.head(torch.cat([(w.unsqueeze(-1) * x).sum(1), x[:, -1]], -1))

def make_batch(tails, idx, K, L):
    """last K*L tokens of each case, split RIGHT-aligned into K chunks of <= L tokens (chunk K-1 is the very end); right-padded; returns ids/mask [B,K,L]"""
    B = len(idx); ids = np.full((B, K, L), PAD, dtype=np.int64); mask = np.zeros((B, K, L), dtype=np.int64)
    for b, i in enumerate(idx):
        t = tails[i][-K * L:]; n = len(t)
        for k in range(K - 1, -1, -1):
            end = n - (K - 1 - k) * L
            if end <= 0: break
            seg = t[max(0, end - L):end]; ids[b, k, :len(seg)] = seg; mask[b, k, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)

def set_backbone_dropout(backbone, p):
    """turn on dropout inside the backbone for fine-tuning (works where the model has nn.Dropout modules, e.g. GPT-2 and T5; a no-op for the others). Returns how many modules were changed."""
    n = 0
    for mod in backbone.modules():
        if isinstance(mod, nn.Dropout): mod.p = p; n += 1
    return n

# ---- windows: window k = the k-th block of L tokens counted back from the end of the case (k=0 is the very end) ----
def n_windows(tail_len, L, W): return max(1, min(W, -(-tail_len // L)))
def get_window(tail, k, L):
    n = len(tail); end = n - k * L
    return tail[-L:] if end <= 0 else tail[max(0, end - L):end]
def make_window_batch(tails, idx, ks, L):
    B = len(idx); ids = np.full((B, 1, L), PAD, dtype=np.int64); mask = np.zeros((B, 1, L), dtype=np.int64)
    for b, (i, k) in enumerate(zip(idx, ks)):
        seg = get_window(tails[i], int(k), L); ids[b, 0, :len(seg)] = seg; mask[b, 0, :len(seg)] = 1
    return torch.from_numpy(ids), torch.from_numpy(mask)


In [ ]:
# ================= CELL 4 — trainer: multi-window fine-tuning, weight averaging (EMA), multi-window inference, rate-matched threshold, submissions =================
import contextlib
USE_BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
def amp_ctx():
    if not AMP: return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float16)
DEFAULT_LR = {"gpt2": 3e-5, "modernbert": 5e-5, "t5": 5e-5, "bilstm": 5e-4}      # gpt2: best of 3e-5 / 1e-4 / 3e-4 in your pilot (the lowest won); the others are untested starting points: use lr_pilot() below
POLICY = "rate_matched"       # test-set decision rule: "rate_matched" (flag the top share of cases as 1, share = the dev positive rate; uses NO test labels) or "dev" (the threshold tuned on dev)
AGG_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]      # recency decay for averaging window scores: weight of window k is a**k (a=0: last window only, a=1: plain mean)

def prf(y, pred):
    f1s, ps, rs = [], [], []
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum()); fn = int(((pred != c) & (y == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0; f = 2 * p * r / (p + r) if p + r else 0.0; ps.append(p); rs.append(r); f1s.append(f)
    return dict(macro_f1=float(np.mean(f1s)), macro_p=float(np.mean(ps)), macro_r=float(np.mean(rs)), f1_0=f1s[0], f1_1=f1s[1], p1=ps[1], r1=rs[1])
def best_threshold(y, prob):
    best_f, best_t = -1.0, 0.5
    for t in sorted(np.arange(0.02, 0.981, 0.01), key=lambda t: abs(t - 0.5)):
        f = prf(y, (prob >= t).astype(int))["macro_f1"]
        if f > best_f + 1e-12: best_f, best_t = f, float(t)
    return best_t, best_f
def roc_auc(y, s):
    r = pd.Series(s).rank().values; n1 = int((y == 1).sum()); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 and n0 else float("nan")
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))
def rm_predict(score, prior):
    """rate matching: the top round(prior*N) scores become class 1. Needs only the scores and the class prior (from dev), never labels of the set being predicted."""
    k = int(round(prior * len(score))); pred = np.zeros(len(score), dtype=int); pred[np.argsort(-score, kind="stable")[:k]] = 1; return pred

@torch.no_grad()
def predict_windows(model, tails, L, W, tok_budget=16384):
    """margin (logit1 - logit0) of each of the last W windows of every case; NaN where the case has fewer windows"""
    model.eval(); n = len(tails); nw = np.array([n_windows(len(t), L, W) for t in tails]); Z = np.full((n, W), np.nan, dtype=np.float32); mb = max(1, tok_budget // L)
    for k in range(W):
        idx_k = np.where(nw > k)[0]
        if len(idx_k) == 0: continue
        order = idx_k[np.argsort([-len(tails[i]) for i in idx_k], kind="stable")]
        for s in range(0, len(order), mb):
            sel = order[s:s + mb]; ids, mask = make_window_batch(tails, sel, [k] * len(sel), L)
            with amp_ctx(): lg = model(ids.to(device), mask.to(device)).float()
            Z[sel, k] = (lg[:, 1] - lg[:, 0]).cpu().numpy()
    return Z
def aggregate(Z, a):
    w = np.array([a ** k for k in range(Z.shape[1])], dtype=np.float32); den = np.where(np.isnan(Z), 0.0, w[None, :]).sum(1)
    return np.nansum(Z * w[None, :], axis=1) / den

class EMA:
    """exponential moving average of the weights; evaluated instead of the raw weights (smoother, usually better at the end of fine-tuning)"""
    def __init__(self, model, decay):
        self.params = [p for p in model.parameters() if p.requires_grad]; self.shadow = [p.detach().clone() for p in self.params]; self.decay = decay; self.backup = None
    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1 + step) / (10 + step)); torch._foreach_mul_(self.shadow, d); torch._foreach_add_(self.shadow, [p.detach() for p in self.params], alpha=1 - d)
    @torch.no_grad()
    def apply(self):
        self.backup = [p.detach().clone() for p in self.params]
        for p, s in zip(self.params, self.shadow): p.copy_(s)
    @torch.no_grad()
    def restore(self):
        for p, b in zip(self.params, self.backup): p.copy_(b)
        self.backup = None

def to_py(x): return x.item() if hasattr(x, "item") else x
def write_submission(path, rids, preds): json.dump([{"id": to_py(r), "pred": int(p)} for r, p in zip(rids, preds)], open(path, "w"))
def validate_submission(path, rids):
    sub = json.load(open(path)); assert isinstance(sub, list) and len(sub) == len(rids), f"{path}: expected {len(rids)} predictions, found {len(sub)}"
    for s, r in zip(sub, rids):
        assert set(s) == {"id", "pred"}, f"bad keys {set(s)}"; assert type(s["pred"]) is int and s["pred"] in (0, 1), f"pred must be the integer 0 or 1, got {s['pred']!r}"; assert s["id"] == to_py(r), f"id mismatch {s['id']!r} vs {to_py(r)!r}"
    assert len({str(s['id']) for s in sub}) == len(sub), "duplicate ids"; return True

def run_experiment(name, model_key, L=512, W=4, epochs=3.0, max_minutes=None, lr=None, head_lr=1e-3, bs=16, wd=0.01, warmup=0.06, p_tail=0.5, ema_decay=0.999, backbone_dropout=0.0, pool="last",
                   dropout=0.1, seed=SEED, max_train_docs=None, dev_cap=None, test_cap=None, evals=4, write=True, policy=None):
    t_start = time.time(); seed_all(seed); rng = np.random.default_rng(seed); policy = policy or POLICY
    assert model_key in MODEL_PATHS, f"'{model_key}' is not available: pre-train it first (found: {list(MODEL_PATHS)})"
    bb, hidden, causal, ctx, fam = load_backbone_any(MODEL_PATHS[model_key]); L = min(L, ctx); lr = lr or DEFAULT_LR[fam]
    nd = set_backbone_dropout(bb, backbone_dropout) if backbone_dropout else 0
    model = DocClassifier(bb, hidden, causal, 1, dropout, pool).to(device); n_params = sum(p.numel() for p in model.parameters())
    tr, dv, te = DATA["train"], DATA["dev"], DATA["test"]
    tr_idx = np.arange(len(tr["tails"])) if not max_train_docs else rng.choice(len(tr["tails"]), min(max_train_docs, len(tr["tails"])), replace=False)
    dv_idx = np.arange(len(dv["tails"])) if not dev_cap else rng.choice(len(dv["tails"]), min(dev_cap, len(dv["tails"])), replace=False); te_idx = np.arange(len(te["tails"])) if not test_cap else np.arange(min(test_cap, len(te["tails"])))
    dv_tails = [dv["tails"][i] for i in dv_idx]; y_dev = dv["labels"][dv_idx]; te_tails = [te["tails"][i] for i in te_idx]; prior = float(np.mean(dv["labels"] == 1))
    sub_n = min(1500, len(dv_idx)); sub = rng.choice(len(dv_idx), sub_n, replace=False); sub_tails = [dv_tails[i] for i in sub]; y_sub = y_dev[sub]
    nwin = np.array([n_windows(len(t), L, W) for t in tr["tails"]])
    cnt = np.bincount(tr["labels"][tr_idx], minlength=2).astype(float); cw = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    mb = max(1, 8192 // L); accum = max(1, math.ceil(bs / mb)); docs_per_step = mb * accum
    groups = [{"params": list(model.backbone.parameters()), "lr": lr}, {"params": [p for n, p in model.named_parameters() if not n.startswith("backbone.")], "lr": head_lr}]
    opt = torch.optim.AdamW(groups, weight_decay=wd); base_lrs = [g["lr"] for g in groups]; scaler = torch.amp.GradScaler("cuda", enabled=AMP and not USE_BF16); ema = EMA(model, ema_decay)
    def sample_ks(idx):
        ks = np.zeros(len(idx), dtype=int)
        for j, i in enumerate(idx):
            if nwin[i] > 1 and rng.random() > p_tail: ks[j] = rng.integers(1, nwin[i])      # an earlier window, with probability 1 - p_tail
        return ks
    def micro(ids_idx):
        ids, mask = make_window_batch(tr["tails"], ids_idx, sample_ks(ids_idx), L); y = torch.from_numpy(tr["labels"][ids_idx]).to(device)
        with amp_ctx(): logits = model(ids.to(device), mask.to(device))
        return F.cross_entropy(logits.float(), y, weight=cw)
    model.train(); times = []
    for j in range(4):                                                            # throughput probe (forward + backward only, no optimizer step); the first is warm-up
        t0 = time.time(); loss = micro(rng.choice(tr_idx, mb)); scaler.scale(loss).backward(); opt.zero_grad(set_to_none=True)
        if device.type == "cuda": torch.cuda.synchronize()
        times.append(time.time() - t0)
    micro_s = float(np.mean(times[1:])); sec_step = micro_s * accum; per_doc_fwd = micro_s / mb / 3.0
    steps = max(10, int(math.ceil(epochs * len(tr_idx) / docs_per_step))); eval_s = ((len(dv_tails) + len(te_tails)) * W + evals * sub_n) * per_doc_fwd
    if max_minutes: steps = min(steps, max(10, int((max_minutes * 60 * 0.92 - eval_s) / sec_step)))
    print(f"[{name}] {fam} | L={L} W={W} p_tail={p_tail} pool={pool} | {n_params/1e6:.0f}M params | lr {lr:g} (head {head_lr:g}) | EMA {ema_decay} | backbone dropout {backbone_dropout} ({nd} modules) | {docs_per_step} cases/step\n"
          f"   plan: {steps} steps = {steps*docs_per_step} cases = {steps*docs_per_step/len(tr_idx):.2f} epochs | {sec_step:.2f}s/step -> ~{steps*sec_step/60:.0f} min training + ~{eval_s/60:.1f} min inference")
    perm, ptr, evals_at, run_loss, hist = rng.permutation(tr_idx), 0, {int(round(steps * (i + 1) / evals)) for i in range(evals)}, 0.0, []
    for step in range(1, steps + 1):
        w = min(step / max(1, int(warmup * steps)), (steps - step + 1) / max(1, steps - int(warmup * steps)), 1.0)
        for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * w
        for _ in range(accum):
            if ptr + mb > len(perm): perm, ptr = rng.permutation(tr_idx), 0
            loss = micro(perm[ptr:ptr + mb]); ptr += mb; scaler.scale(loss / accum).backward(); run_loss += loss.item() / accum
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); ema.update(step)
        if step % max(1, steps // 12) == 0: print(f"   step {step}/{steps} | train loss {run_loss/max(1, steps//12):.4f} | {(time.time()-t_start)/60:.1f} min", flush=True); run_loss = 0.0
        if step in evals_at:                                                       # monitoring only: EMA weights, last window, dev subset (no snapshot selection: the final EMA weights are used)
            ema.apply(); z = predict_windows(model, sub_tails, L, 1)[:, 0]; ema.restore(); model.train(); a_ = roc_auc(y_sub, z); t_, f_ = best_threshold(y_sub, sigmoid(z)); hist.append((step, round(a_, 4), round(f_, 4)))
            print(f"   [dev subset {sub_n}, EMA, last window] step {step}: AUC {a_:.4f} | macro-F1 {f_:.4f}", flush=True)
    ema.apply()                                                                    # final model = EMA weights
    Zd, Zt = predict_windows(model, dv_tails, L, W), predict_windows(model, te_tails, L, W)
    aucs = {a: roc_auc(y_dev, aggregate(Zd, a)) for a in AGG_GRID}; a_best = max(AGG_GRID, key=lambda a: (round(aucs[a], 6), -a)); m_dev, m_te = aggregate(Zd, a_best), aggregate(Zt, a_best); p_dev, p_te = sigmoid(m_dev), sigmoid(m_te)
    thr, f_tuned = best_threshold(y_dev, p_dev); at05 = prf(y_dev, (p_dev >= 0.5).astype(int)); rm = prf(y_dev, rm_predict(p_dev, prior)); tuned = prf(y_dev, (p_dev >= thr).astype(int))
    pred_te = rm_predict(p_te, prior) if policy == "rate_matched" else (p_te >= thr).astype(int)
    res = dict(name=name, model=model_key, family=fam, L=L, W=W, p_tail=p_tail, pool=pool, params_M=round(n_params / 1e6, 1), lr=lr, epochs=round(steps * docs_per_step / len(tr_idx), 2), steps=steps, agg_a=a_best,
               dev_auc=aucs[a_best], dev_auc_last_window=aucs[0.0], dev_f1_ratematched=rm["macro_f1"], dev_f1_tuned=f_tuned, dev_f1_at_0p5=at05["macro_f1"], threshold=thr, dev_p1=tuned["p1"], dev_r1=tuned["r1"], dev_cases=len(y_dev),
               test_pos_rate_at_dev_thr=float(np.mean(p_te >= thr)), policy=policy, minutes=round((time.time() - t_start) / 60, 1), history=str(hist))
    print(f"[{name}] DEV: AUC {aucs[a_best]:.4f} (last window only: {aucs[0.0]:.4f}; chosen recency decay a={a_best}) | macro-F1 rate-matched {rm['macro_f1']:.4f} | tuned {f_tuned:.4f} (thr {thr:.2f}) | at 0.5 {at05['macro_f1']:.4f} | {res['minutes']} min\n"
          f"   test: share predicted positive at the dev threshold = {res['test_pos_rate_at_dev_thr']:.3f} (dev positive rate {prior:.3f}) -> policy '{policy}' used for the submission file")
    if write:
        rd = WORK / "runs" / name; rd.mkdir(exist_ok=True); np.save(rd / "dev_Z.npy", Zd); np.save(rd / "test_Z.npy", Zt); np.save(rd / "dev_probs.npy", p_dev); np.save(rd / "test_probs.npy", p_te); json.dump(res, open(rd / "result.json", "w"), indent=1)
        rids = [te["rids"][i] for i in te_idx]; write_submission(SUBDIR / f"{name}.json", rids, pred_te); validate_submission(SUBDIR / f"{name}.json", rids)
        row = pd.DataFrame([res]); led = pd.concat([pd.read_csv(LEDGER).query('name != @name'), row]) if LEDGER.exists() else row; led.to_csv(LEDGER, index=False)
        print(f"   saved {SUBDIR / (name + '.json')} ({len(rids)} predictions, format validated) and the ledger row")
    del model, opt, bb, ema; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return res

def lr_pilot(model_key, grid=(1.5e-5, 3e-5, 1e-4), minutes=10, L=512, **kw):
    """short runs on a subset; picks the learning rate by dev AUC (threshold-free, less noisy than F1). Differences below about 0.01 AUC are noise: prefer the middle value then."""
    out = {}
    for lr_ in grid:
        r = run_experiment(f"pilot_{model_key}_{lr_:g}", model_key, L=L, W=1, epochs=99, max_minutes=minutes, lr=lr_, max_train_docs=10000, dev_cap=1500, test_cap=50, evals=1, write=False, ema_decay=0.99, **kw); out[lr_] = r["dev_auc"]
    best = max(out, key=out.get); print("\nLR pilot (dev AUC):", {f"{k:g}": round(v, 4) for k, v in out.items()}, "-> best", f"{best:g}"); return best


In [ ]:
# ================= CELL 5 — SMOKE TEST (~4 min): small data, windows + EMA + rate matching all exercised, writes nothing. Must pass before the real runs. =================
res = run_experiment("smoke", "gpt2_new", L=512, W=2, epochs=99, max_minutes=2.5, max_train_docs=600, dev_cap=300, test_cap=40, evals=1, write=False, ema_decay=0.98)
assert np.isfinite(res["dev_auc"]); print("\nsmoke test OK. Expect a LOW score here (about 600 training cases); the plan lines above show the real speed, so you can size the experiments below.")


In [ ]:
# ================= CELL 6 — EXPERIMENTS. Run each cell you can; every run writes its scores, probabilities and a validated submission file, and adds a ledger row. =================
# What is new compared with the first notebook: 3 epochs instead of 1.3, weight averaging (EMA), several windows from the END of each case (random window during training, all windows averaged at inference),
# a rate-matched decision threshold, and a controlled A/B (X1 vs X2) so the multi-window idea is measured, not assumed.
# Every run uses the same recipe, so differences are due to the model/strategy. Differences below about 0.01 AUC or 0.02 macro-F1 are within noise on 3,505 dev cases.


In [ ]:
# X1: new GPT-2, multi-window training (50% last window, 50% an earlier window among the last 2,048 tokens) + 4-window averaging at inference   (~75 min on a T4)
X1 = run_experiment("X1_gpt2_win4", "gpt2_new", L=512, W=4, epochs=3, lr=3e-5, p_tail=0.5)


In [ ]:
# X2: CONTROL for X1: always trains on the last window; inference still scores all 4 windows, so the table shows what window training and window averaging each add   (~75 min)
X2 = run_experiment("X2_gpt2_tail", "gpt2_new", L=512, W=4, epochs=3, lr=3e-5, p_tail=1.0)


In [ ]:
# optional: does averaging the last-token and mean-pooled vectors help the causal model? (same recipe as X1 except pool)   (~75 min)
X1b = run_experiment("X1b_gpt2_win4_meanlast", "gpt2_new", L=512, W=4, epochs=3, lr=3e-5, p_tail=0.5, pool="meanlast")


In [ ]:
# encoder-only model (needs the pre-trained encoder dataset attached). The learning rate is untested for it: run the pilot first (about 30 min), then the real run.
LR_ENC = lr_pilot("modernbert", grid=(2e-5, 5e-5, 1.5e-4), minutes=10)
X3 = run_experiment("X3_encoder_win4", "modernbert", L=512, W=4, epochs=3, lr=LR_ENC, p_tail=0.5)


In [ ]:
# encoder of the encoder-decoder model (T5): same procedure
LR_T5 = lr_pilot("t5", grid=(2e-5, 5e-5, 1.5e-4), minutes=10)
X4 = run_experiment("X4_t5enc_win4", "t5", L=512, W=4, epochs=3, lr=LR_T5, p_tail=0.5)


In [ ]:
# recurrent model (BiLSTM): longer windows are cheap for it; the learning rate is untested
LR_RNN = lr_pilot("bilstm", grid=(2e-4, 5e-4, 1e-3), minutes=10, L=1024)
X5 = run_experiment("X5_bilstm_win4", "bilstm", L=1024, W=4, epochs=3, lr=LR_RNN, p_tail=0.5)


In [ ]:
# the OLD 3,000-step GPT-2 (trained on 4,096-token blocks): 2,048-token windows, last 4,096 tokens in total
X6 = run_experiment("X6_gpt2old_win2", "gpt2_old", L=2048, W=2, epochs=2, lr=3e-5, p_tail=0.5)


In [ ]:
# ================= CELL 7 — choose the six submissions, build the ensemble, write prediction.json, validate, package =================
import shutil
led = pd.read_csv(LEDGER); led = led[~led.name.str.startswith("pilot")].sort_values("dev_auc", ascending=False).reset_index(drop=True)
cols = ["name", "model", "L", "W", "p_tail", "epochs", "agg_a", "dev_auc", "dev_auc_last_window", "dev_f1_ratematched", "dev_f1_tuned", "test_pos_rate_at_dev_thr", "minutes"]; print(led[cols].round(4).to_string(index=False))
rids = DATA["test"]["rids"]; y_dev = DATA["dev"]["labels"]; prior = float(np.mean(y_dev == 1))
# ---- the six submissions required by the rules: the best experiment of every distinct Phase 1 model first, then the next best ----
FINAL = None                                    # or set a list of six experiment names by hand
if FINAL is None:
    FINAL = list(led.drop_duplicates("model").head(6).name)
    for n in led.name:
        if len(FINAL) >= 6: break
        if n not in FINAL: FINAL.append(n)
assert len(FINAL) == 6, f"need six experiments, the ledger has {len(led)}: run more experiments"
fams = led.set_index("name").loc[FINAL, "model"].tolist(); print("\nFINAL SIX:", FINAL, "\nPhase 1 models covered:", sorted(set(fams)))
if not {"gpt2_new", "modernbert", "t5", "bilstm"} <= set(fams): print("WARNING: the rules require experiments with ALL FOUR Phase 1 architectures; missing:", sorted({"gpt2_new", "modernbert", "t5", "bilstm"} - set(fams)))
FINAL_DIR = WORK / "final_six"; shutil.rmtree(FINAL_DIR, ignore_errors=True); FINAL_DIR.mkdir()
for k, n in enumerate(FINAL, 1): validate_submission(SUBDIR / f"{n}.json", rids); (FINAL_DIR / f"submission_{k}_{n}.json").write_text((SUBDIR / f"{n}.json").read_text())
# ---- ensemble: average of dev-standardised margins; the number of models is chosen on dev AUC (threshold-free), preferring MORE models when within 0.002 ----
def margins(n):
    pd_, pt_ = np.load(WORK / "runs" / n / "dev_probs.npy"), np.load(WORK / "runs" / n / "test_probs.npy"); lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6))); md, mt = lg(pd_), lg(pt_); s = md.std() + 1e-9
    return md / s, mt / s
M = {n: margins(n) for n in led.name}; order = list(led.name); aucs_k = []
for k in range(1, len(order) + 1): aucs_k.append(roc_auc(y_dev, np.mean([M[n][0] for n in order[:k]], 0)))
best_auc = max(aucs_k); k_ens = max(k for k, a in enumerate(aucs_k, 1) if a >= best_auc - 0.002); ens_names = order[:k_ens]
z_dev = np.mean([M[n][0] for n in ens_names], 0); z_te = np.mean([M[n][1] for n in ens_names], 0)
print("\nensemble dev AUC by number of models (best first):", [round(a, 4) for a in aucs_k], f"\n-> using the top {k_ens}: {ens_names}")
e_rm = prf(y_dev, rm_predict(z_dev, prior)); t_e, f_e = best_threshold(y_dev, sigmoid(z_dev * 1.0))
print(f"ensemble DEV: AUC {roc_auc(y_dev, z_dev):.4f} | macro-F1 rate-matched {e_rm['macro_f1']:.4f} | best single run: AUC {led.dev_auc.max():.4f}, rate-matched macro-F1 {led.dev_f1_ratematched.max():.4f}")
pred_ens = rm_predict(z_te, prior); write_submission(Path("/kaggle/working/prediction_ensemble.json"), rids, pred_ens); validate_submission(Path("/kaggle/working/prediction_ensemble.json"), rids)
best_single = led.iloc[0]["name"]; shutil.copy(SUBDIR / f"{best_single}.json", Path("/kaggle/working/prediction_best_single.json"))
use_ens = roc_auc(y_dev, z_dev) >= led.dev_auc.max() + 0.002; shutil.copy("/kaggle/working/prediction_ensemble.json" if use_ens else str(SUBDIR / f"{best_single}.json"), "/kaggle/working/prediction.json"); validate_submission(Path("/kaggle/working/prediction.json"), rids)
print(f"\nprediction.json = {'the ensemble' if use_ens else best_single} (chosen on dev AUC only; the leaderboard plays no part in the choice) | positives predicted on test: {np.mean([p['pred'] for p in json.load(open('/kaggle/working/prediction.json'))]):.3f}")
with zipfile.ZipFile("/kaggle/working/cjpe_results_v2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for n in led.name:
        for f in ("dev_Z.npy", "test_Z.npy", "dev_probs.npy", "test_probs.npy", "result.json"): z.write(WORK / "runs" / n / f, f"runs/{n}/{f}")
        z.write(SUBDIR / f"{n}.json", f"submissions/{n}.json")
    z.write(LEDGER, "ledger.csv")
    for p in sorted(FINAL_DIR.glob("*.json")): z.write(p, "final_six/" + p.name)
print("files in /kaggle/working: prediction.json, prediction_ensemble.json, prediction_best_single.json, final_six/ (six files), cjpe_results_v2.zip")
print("RULES OF THE ROAD: choose by DEV only; look at the leaderboard once per final candidate; differences under ~0.02 on 1,517 test cases are noise (one standard error is about 0.012).")


In [ ]:
# ================= CELL 8 — DIAGNOSTICS on dev for the ensemble above (where does it fail?) =================
pred_d = rm_predict(z_dev, prior); lens = DATA["dev"]["lens"]; qs = np.quantile(lens, [0, .25, .5, .75, 1.0]); rows = []
for lo, hi in zip(qs[:-1], qs[1:]):
    sel = (lens >= lo) & (lens <= hi); r = prf(y_dev[sel], pred_d[sel]); rows.append(dict(tokens=f"{int(lo)}-{int(hi)}", cases=int(sel.sum()), macro_f1=round(r["macro_f1"], 3), acc=round(float((pred_d[sel] == y_dev[sel]).mean()), 3), auc=round(roc_auc(y_dev[sel], z_dev[sel]), 3)))
print("dev performance by case length (token quartiles):"); print(pd.DataFrame(rows).to_string(index=False)); al = prf(y_dev, pred_d)
print(f"\noverall (rate-matched): macro-F1 {al['macro_f1']:.4f} | class-1 precision {al['p1']:.4f} recall {al['r1']:.4f} | class-0 F1 {al['f1_0']:.4f} class-1 F1 {al['f1_1']:.4f}")
conf = pd.crosstab(pd.Series(y_dev, name="true"), pd.Series(pred_d, name="predicted")); print("\nconfusion matrix (dev):"); print(conf.to_string())
print("\nrun-to-run agreement on dev (correlation of standardised margins) - low values = diverse models, good for the ensemble:"); print(pd.DataFrame({n: M[n][0] for n in order}).corr().round(2).to_string())
